In [13]:
import numpy as np
import pandas as pd
import json

In [14]:
!ls ~/LLM-RAG-Lora/Lora-Llama-3/dataset/input/finance/QA/csv

10k_test.csv  Financial-QA-10k.csv


In [15]:
data = pd.read_csv('QA/csv/Financial-QA-10k.csv')

In [16]:
data.head()

,question,answer,context
0,What area did NVIDIA initially focus on before...,NVIDIA initially focused on PC graphics.,"Since our original focus on PC graphics, we ha..."
1,What are some of the recent applications of GP...,Recent applications of GPU-powered deep learni...,Some of the most recent applications of GPU-po...
2,What significant invention did NVIDIA create i...,NVIDIA invented the GPU in 1999.,Our invention of the GPU in 1999 defined moder...
3,How does NVIDIA's platform strategy contribute...,NVIDIA's platform strategy brings together har...,"NVIDIA has a platform strategy, bringing toget..."
4,What does NVIDIA's CUDA programming model enable?,NVIDIA's CUDA programming model opened the par...,With our introduction of the CUDA programming ...


In [17]:
# data.replace({'Sentiment': {1: 'positive', 0: 'negative or neutral'}}, inplace=True)
random_state = 42
test_data = data.sample(frac=0.2, random_state=random_state)
train_data = data.drop(test_data.index)

In [18]:
# 构造 LoRA / Alpaca 格式数据
lora_data = []
for _, row in train_data.iterrows():
    lora_data.append({
        "instruction": "Please answer the financial or operational question based on the provided context.",
        "input": f"Question: {row['question']}\n\nContext: {row['context']}",
        "output": row["answer"]
    })

# 保存为 JSON 格式
with open('QA/json/10k_qa.json', 'w', encoding='utf-8') as f:
    json.dump(lora_data, f, ensure_ascii=False, indent=2)

In [19]:
test_data.reset_index(inplace=True, drop=True)

In [20]:
# 原始文件路径
input_file = "sentiment/json/stock_sentiment.json"
output_file = input_file

# 要替换的新instruction内容
new_instruction = "Determine whether the sentiment of the financial sentence is positive, negative, or neutral."

# 读取 JSON 数据
with open(input_file, "r", encoding="utf-8") as f:
    data = json.load(f)

# 修改 instruction 字段
for item in data:
    if "instruction" in item:
        item["instruction"] = new_instruction

# 写入新文件
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

In [16]:
import numpy as np
import pandas as pd
import json
import os
import random

file_list = [
    "QA/json/10k_qa.json",
    "sentiment/json/stock_sentiment.json",  # 第二个
    "sentiment/json/news_sentiment.json",
    "sentiment/json/sentence_sentiment.json"
]

merged_data = []
test_data = []

# 设置随机种子以保证降采样结果可复现（可选）
random.seed(42)

for idx, file_name in enumerate(file_list):
    
    with open(file_name, "r", encoding="utf-8") as f:
        data = json.load(f)
        print(f"{file_name} size: {os.path.getsize(file_name) / (1024 * 1024):.2f} MB")
    if idx == 1:  # 针对第二个文件 stock_sentiment.json
        random.shuffle(data)
        half = int(len(data) * 0.6)
        sampled = data[:half]   # 用于训练
        removed = data[half:]   # 用于测试

        merged_data.extend(sampled)
        print(len(merged_data))        

    elif idx == 2:  # 针对第二个文件 stock_sentiment.json
        random.shuffle(data)
        half = int(len(data) * 0.1)
        sampled = data[:half]   # 用于训练
        removed = data[half:]   # 用于测试

        merged_data.extend(sampled)
        test_data.extend(removed)

    elif idx == 3:  # 针对第二个文件 stock_sentiment.json
        random.shuffle(data)
        half = int(len(data) * 0.1)
        sampled = data[:half]   # 用于训练
        removed = data[half:]   # 用于测试

        merged_data.extend(sampled)
        test_data.extend(removed)
# 保存合并后的结果
output_file = "merged.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(merged_data, f, indent=2, ensure_ascii=False)

test_path = "sentiment/json/sentiment_test.json"
with open(test_path, "w", encoding="utf-8") as test_file:
    json.dump(removed, test_file, ensure_ascii=False, indent=2)

# 获取文件大小（单位 MB）
size_mb = os.path.getsize(output_file) / (1024 * 1024)
print(f"✅ 合并完成！输出文件：{output_file}")
print(f"📦 文件大小：{size_mb:.2f} MB")
# 检查 Financial-QA-10k.csv 是否被合并进 merged.json
qa_csv_path = "QA/csv/Financial-QA-10k.csv"
qa_df = pd.read_csv(qa_csv_path)

# 检查 merged_data 是否包含 qa_df 的内容（以 question 字段为例）
qa_questions = set(qa_df["question"].astype(str))
merged_inputs = set(item["input"] for item in merged_data if "input" in item)


QA/json/10k_qa.json size: 3.24 MB
sentiment/json/stock_sentiment.json size: 20.62 MB
52200
sentiment/json/news_sentiment.json size: 8.11 MB
sentiment/json/sentence_sentiment.json size: 1.57 MB
✅ 合并完成！输出文件：merged.json
📦 文件大小：13.34 MB


In [18]:
import csv
# 读取 JSON 文件
with open("sentiment/json/sentiment_test.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# 写入 CSV 文件
with open("sentiment/csv/test_data.csv", "w", encoding="utf-8", newline='') as csvfile:
    writer = csv.writer(csvfile)
    # 写入标题
    writer.writerow(["input", "output"])

    # 遍历每条数据，提取 input 和 output 字段
    for item in data:
        writer.writerow([item.get("input", ""), item.get("output", "")])

In [22]:
from datasets import Dataset
df = pd.read_json('merged.json')
ds = Dataset.from_pandas(df)

In [23]:
ds

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 125426
})

In [24]:
from datasets import Dataset

# 统计缺失数量的函数
def count_missing(example):
    return {
        "instruction_missing": example["instruction"] is None or example["instruction"] == "",
        "input_missing": example["input"] is None or example["input"] == "",
        "output_missing": example["output"] is None or example["output"] == ""
    }

# 应用到整个数据集（batched=False 单条处理）
missing_flags = ds.map(count_missing)

# 汇总缺失数量
instruction_missing = sum(missing_flags["instruction_missing"])
input_missing = sum(missing_flags["input_missing"])
output_missing = sum(missing_flags["output_missing"])

print(f"📊 缺失统计：")
print(f"  instruction 缺失条数: {instruction_missing}")
print(f"  input 缺失条数     : {input_missing}")
print(f"  output 缺失条数    : {output_missing}")


Map: 100%|██████████| 125426/125426 [00:05<00:00, 21689.89 examples/s]


📊 缺失统计：
  instruction 缺失条数: 0
  input 缺失条数     : 15
  output 缺失条数    : 0


In [6]:
import numpy as np
import pandas as pd
dija = pd.read_csv('sentiment/csv/djia_news copy.csv')

In [9]:
dija.head()

,Label,Ticker,Headline
0,0,MMM,Employer who stole nearly $3M in wages from 15...
1,1,MMM,Huge new Facebook data leak exposed intimate d...
2,0,MMM,A campaign has accelerated to turn a disused r...
3,1,MMM,Google launches global human trafficking helpl...
4,1,MMM,Over 3m Saudi Women Don’t Have ID Cards; Saudi...


In [10]:
dija.drop(columns=['Ticker'], inplace=True)

In [11]:
dija.head()

,Label,Headline
0,0,Employer who stole nearly $3M in wages from 15...
1,1,Huge new Facebook data leak exposed intimate d...
2,0,A campaign has accelerated to turn a disused r...
3,1,Google launches global human trafficking helpl...
4,1,Over 3m Saudi Women Don’t Have ID Cards; Saudi...


In [ ]:
dija

In [13]:
label_mapping = {
    0: 'negative',
    1: 'positive',
    2: 'neutral'
}

# 使用map函数进行替换
dija['Label'] = dija['Label'].map(label_mapping)


In [15]:
dija.to_csv('dija_news.csv', index=False)